In [1]:
# 01. IMPORT LIBRARIES

import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
# 02. SET UP PROJECT PATHS

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
IBTRACS_DIR = DATA_DIR / "raw" / "ibtracs"
CLEANED_DIR = DATA_DIR / "interim" / "cleaned_tracks"

NI_FILE = IBTRACS_DIR / "ibtracs.NI.list.v04r01.csv"

print("Project root:", PROJECT_ROOT)
print("Input file:", NI_FILE)
print("Output folder:", CLEANED_DIR)

Project root: d:\SIH\tropical-cyclone-ai
Input file: d:\SIH\tropical-cyclone-ai\data\raw\ibtracs\ibtracs.NI.list.v04r01.csv
Output folder: d:\SIH\tropical-cyclone-ai\data\interim\cleaned_tracks


In [3]:
# 03. LOAD RAW IBTRACS DATA

ni_df = pd.read_csv(
    NI_FILE,
    skiprows=[1],
    low_memory=False
)

print("Dataset loaded successfully.")
print("Shape:", ni_df.shape)

Dataset loaded successfully.
Shape: (62848, 174)


In [4]:
# 04. CREATE WORKING COPY

clean_df = ni_df.copy()

print("Working copy created.")
print("Shape:", clean_df.shape)

Working copy created.
Shape: (62848, 174)


In [5]:
# 05. STANDARDIZE COLUMN NAMES

clean_df.columns = (
    clean_df.columns
    .str.strip()
    .str.upper()
)

print("Column names standardized.")

Column names standardized.


In [6]:
# 06. STANDARDIZE MISSING VALUES

clean_df = clean_df.replace(
    r"^\s*$",
    np.nan,
    regex=True
)

print("Blank and whitespace-only values converted to NaN.")

Blank and whitespace-only values converted to NaN.


In [7]:
# 07. CHECK MISSING VALUES

missing_summary = (
    clean_df.isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing_summary.head(20))

NADI_LAT           62848
NADI_LON           62848
NADI_CAT           62848
NADI_WIND          62848
NADI_PRES          62848
WELLINGTON_LAT     62848
WELLINGTON_LON     62848
WELLINGTON_WIND    62848
WELLINGTON_PRES    62848
DS824_PRES         62848
NEUMANN_PRES       62848
MLC_LAT            62848
NEUMANN_WIND       62848
TD9636_PRES        62848
USA_SEARAD_NW      62848
USA_SEAHGT         62848
USA_SEARAD_NE      62848
USA_SEARAD_SE      62848
NEUMANN_CLASS      62848
REUNION_LON        62848
dtype: int64

In [8]:
# 08. CONVERT CORE DATA TYPES

clean_df["ISO_TIME"] = pd.to_datetime(
    clean_df["ISO_TIME"],
    errors="coerce"
)

clean_df["LAT"] = pd.to_numeric(
    clean_df["LAT"],
    errors="coerce"
)

clean_df["LON"] = pd.to_numeric(
    clean_df["LON"],
    errors="coerce"
)

clean_df["SEASON"] = pd.to_numeric(
    clean_df["SEASON"],
    errors="coerce"
)

print("Core data types converted.")

Core data types converted.


In [9]:
# 09. CLEAN TEXT FIELDS

text_columns = [
    "SID",
    "BASIN",
    "SUBBASIN",
    "NAME",
    "NATURE",
    "TRACK_TYPE"
]

for col in text_columns:
    clean_df[col] = (
        clean_df[col]
        .astype("string")
        .str.strip()
    )

print("Text fields cleaned.")

Text fields cleaned.


In [10]:
# 10. CONVERT INTENSITY FIELDS

intensity_columns = [
    "WMO_WIND",
    "WMO_PRES",
    "USA_WIND",
    "USA_PRES",
    "NEWDELHI_WIND",
    "NEWDELHI_PRES",
    "TOKYO_WIND",
    "TOKYO_PRES"
]

for col in intensity_columns:
    clean_df[col] = pd.to_numeric(
        clean_df[col],
        errors="coerce"
    )

print("Intensity and pressure fields converted to numeric.")

Intensity and pressure fields converted to numeric.


In [11]:
# 11. VERIFY DATA TYPES

display(
    clean_df[
        [
            "SID",
            "SEASON",
            "ISO_TIME",
            "LAT",
            "LON",
            "WMO_WIND",
            "WMO_PRES",
            "USA_WIND",
            "USA_PRES",
            "NEWDELHI_WIND",
            "NEWDELHI_PRES"
        ]
    ].dtypes
)

SID                      string
SEASON                    int64
ISO_TIME         datetime64[us]
LAT                     float64
LON                     float64
WMO_WIND                float64
WMO_PRES                float64
USA_WIND                float64
USA_PRES                float64
NEWDELHI_WIND           float64
NEWDELHI_PRES           float64
dtype: object

In [12]:
# 12. CHECK INVALID CORE VALUES

print("Invalid SID:", clean_df["SID"].isna().sum())
print("Invalid time:", clean_df["ISO_TIME"].isna().sum())
print("Invalid latitude:", clean_df["LAT"].isna().sum())
print("Invalid longitude:", clean_df["LON"].isna().sum())
print("Invalid season:", clean_df["SEASON"].isna().sum())

Invalid SID: 0
Invalid time: 0
Invalid latitude: 0
Invalid longitude: 0
Invalid season: 0


In [13]:
# 13. REMOVE ROWS MISSING ESSENTIAL FIELDS
# SID + ISO_TIME + LAT + LON

essential_columns = [
    "SID",
    "ISO_TIME",
    "LAT",
    "LON"
]

before = len(clean_df)

clean_df = clean_df.dropna(
    subset=essential_columns
).copy()

after = len(clean_df)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 62848
Rows after: 62848
Rows removed: 0


In [14]:
# 14. VALIDATE COORDINATE RANGES

invalid_lat = (
    (clean_df["LAT"] < -90) |
    (clean_df["LAT"] > 90)
)

invalid_lon = (
    (clean_df["LON"] < -180) |
    (clean_df["LON"] > 180)
)

print("Invalid latitude rows:", invalid_lat.sum())
print("Invalid longitude rows:", invalid_lon.sum())

Invalid latitude rows: 0
Invalid longitude rows: 0


In [15]:
# 15. REMOVE INVALID COORDINATES

clean_df = clean_df[
    ~invalid_lat & ~invalid_lon
].copy()

print("Dataset shape after coordinate validation:", clean_df.shape)

Dataset shape after coordinate validation: (62848, 174)


In [16]:
# 16. CHECK DUPLICATE CYCLONE-TIME OBSERVATIONS

duplicate_track_points = clean_df.duplicated(
    subset=["SID", "ISO_TIME"]
).sum()

print(
    "Duplicate SID + ISO_TIME observations:",
    duplicate_track_points
)

Duplicate SID + ISO_TIME observations: 0


In [17]:
# 17. REMOVE DUPLICATE CYCLONE-TIME OBSERVATIONS

before = len(clean_df)

clean_df = clean_df.drop_duplicates(
    subset=["SID", "ISO_TIME"],
    keep="first"
).copy()

after = len(clean_df)

print("Rows before:", before)
print("Rows after:", after)
print("Duplicate rows removed:", before - after)

Rows before: 62848
Rows after: 62848
Duplicate rows removed: 0


In [18]:
# 18. CHECK INTENSITY RANGES

wind_columns = [
    "WMO_WIND",
    "USA_WIND",
    "NEWDELHI_WIND",
    "TOKYO_WIND"
]

pressure_columns = [
    "WMO_PRES",
    "USA_PRES",
    "NEWDELHI_PRES",
    "TOKYO_PRES"
]

print("Wind ranges:")

for col in wind_columns:
    print(
        col,
        "min =", clean_df[col].min(),
        "max =", clean_df[col].max()
    )

print("\nPressure ranges:")

for col in pressure_columns:
    print(
        col,
        "min =", clean_df[col].min(),
        "max =", clean_df[col].max()
    )

Wind ranges:
WMO_WIND min = 3.0 max = 140.0
USA_WIND min = 10.0 max = 150.0
NEWDELHI_WIND min = 3.0 max = 140.0
TOKYO_WIND min = 35.0 max = 120.0

Pressure ranges:
WMO_PRES min = 890.0 max = 1012.0
USA_PRES min = 898.0 max = 1014.0
NEWDELHI_PRES min = 912.0 max = 1011.0
TOKYO_PRES min = 890.0 max = 1012.0


In [19]:
# 19. FINAL CORE FIELD CHECK

core_columns = [
    "SID",
    "ISO_TIME",
    "LAT",
    "LON"
]

display(
    clean_df[core_columns]
    .isna()
    .sum()
    .to_frame("Missing")
)

,Missing
SID,0
ISO_TIME,0
LAT,0
LON,0


In [21]:
# 20. CREATE CANONICAL WIND FIELD

clean_df["WIND_KTS"] = (
    clean_df["USA_WIND"]
    .combine_first(clean_df["NEWDELHI_WIND"])
    .combine_first(clean_df["WMO_WIND"])
    .combine_first(clean_df["TOKYO_WIND"])
)

print("Observation with canonical wind:", clean_df["WIND_KTS"].notna().sum())
print("Observation without canonical wind:", clean_df["WIND_KTS"].isna().sum())

Observation with canonical wind: 19608
Observation without canonical wind: 43240


In [23]:
# 21. CREATE CANONICAL PRESSURE FIELD

clean_df["PRESSURE_MB"] = (
    clean_df["USA_PRES"]
    .combine_first(clean_df["NEWDELHI_PRES"])
    .combine_first(clean_df["WMO_PRES"])
    .combine_first(clean_df["TOKYO_PRES"])
)

print(
    "Observations with canonical pressure:",
    clean_df["PRESSURE_MB"].notna().sum()
)

print(
    "Observations without canonical pressure:",
    clean_df["PRESSURE_MB"].isna().sum()
)

Observations with canonical pressure: 14194
Observations without canonical pressure: 48654


In [24]:
# 22. RECORD INTENSITY SOURCES

clean_df["WIND_SOURCE"] = np.select(
    [
        clean_df["USA_WIND"].notna(),
        clean_df["NEWDELHI_WIND"].notna(),
        clean_df["WMO_WIND"].notna(),
        clean_df["TOKYO_WIND"].notna()
    ],
    [
        "USA",
        "NEWDELHI",
        "WMO",
        "TOKYO"
    ],
    default="MISSING"
)

clean_df["PRESSURE_SOURCE"] = np.select(
    [
        clean_df["USA_PRES"].notna(),
        clean_df["NEWDELHI_PRES"].notna(),
        clean_df["WMO_PRES"].notna(),
        clean_df["TOKYO_PRES"].notna()
    ],
    [
        "USA",
        "NEWDELHI",
        "WMO",
        "TOKYO"
    ],
    default="MISSING"
)

print("Source columns created.")

Source columns created.


In [25]:
# 23. CHECK CANONICAL FIELDS

display(
    clean_df[
        [
            "SID",
            "ISO_TIME",
            "LAT",
            "LON",
            "WIND_KTS",
            "WIND_SOURCE",
            "PRESSURE_MB",
            "PRESSURE_SOURCE"
        ]
    ].head(10)
)

,SID,ISO_TIME,LAT,LON,WIND_KTS,WIND_SOURCE,PRESSURE_MB,PRESSURE_SOURCE
0,1842298N11080,1842-10-25 03:00:00,10.9,80.3,NaN,MISSING,NaN,MISSING
1,1842298N11080,1842-10-25 06:00:00,10.9,79.8,NaN,MISSING,NaN,MISSING
2,1842298N11080,1842-10-25 09:00:00,10.8,79.4,NaN,MISSING,NaN,MISSING
3,1842298N11080,1842-10-25 12:00:00,10.8,78.9,NaN,MISSING,NaN,MISSING
4,1842298N11080,1842-10-25 15:00:00,10.8,78.4,NaN,MISSING,NaN,MISSING
5,1842298N11080,1842-10-25 18:00:00,10.8,77.9,NaN,MISSING,NaN,MISSING
6,1842298N11080,1842-10-25 21:00:00,10.8,77.4,NaN,MISSING,NaN,MISSING
7,1842298N11080,1842-10-26 00:00:00,10.8,76.9,NaN,MISSING,NaN,MISSING
8,1842298N11080,1842-10-26 03:00:00,10.8,76.4,NaN,MISSING,NaN,MISSING
9,1842298N11080,1842-10-26 06:00:00,10.8,75.8,NaN,MISSING,NaN,MISSING


In [26]:
# 24. SELECT USEFUL TRACK COLUMNS

track_columns = [
    "SID",
    "SEASON",
    "BASIN",
    "SUBBASIN",
    "NAME",
    "ISO_TIME",
    "NATURE",
    "LAT",
    "LON",
    "WIND_KTS",
    "WIND_SOURCE",
    "PRESSURE_MB",
    "PRESSURE_SOURCE"
]

clean_tracks = clean_df[track_columns].copy()

print("Clean track dataset shape:", clean_tracks.shape)

Clean track dataset shape: (62848, 13)


In [27]:
# 25. SORT TRACK DATA

clean_tracks = clean_tracks.sort_values(
    ["SID", "ISO_TIME"]
).reset_index(drop=True)

print("Track data sorted by cyclone and time.")

Track data sorted by cyclone and time.


In [28]:
# 26. FINAL QUALITY CHECK

print("Rows:", len(clean_tracks))
print("Columns:", len(clean_tracks.columns))
print("Duplicate SID + ISO_TIME:",
      clean_tracks.duplicated(
          subset=["SID", "ISO_TIME"]
      ).sum())

print("\nMissing values:")
display(
    clean_tracks.isna().sum()
    .sort_values(ascending=False)
)

Rows: 62848
Columns: 13
Duplicate SID + ISO_TIME: 0

Missing values:


PRESSURE_MB        48654
WIND_KTS           43240
BASIN                482
SUBBASIN             449
SID                    0
NAME                   0
SEASON                 0
ISO_TIME               0
NATURE                 0
LON                    0
LAT                    0
WIND_SOURCE            0
PRESSURE_SOURCE        0
dtype: int64

In [31]:
# 27. PREVIEW CLEANED TRACK DATA

display(clean_tracks.head(10))

,SID,SEASON,BASIN,SUBBASIN,NAME,ISO_TIME,NATURE,LAT,LON,WIND_KTS,WIND_SOURCE,PRESSURE_MB,PRESSURE_SOURCE
0,1842298N11080,1842,NI,BB,UNNAMED,1842-10-25 03:00:00,NR,10.9,80.3,NaN,MISSING,NaN,MISSING
1,1842298N11080,1842,NI,BB,UNNAMED,1842-10-25 06:00:00,NR,10.9,79.8,NaN,MISSING,NaN,MISSING
2,1842298N11080,1842,NI,BB,UNNAMED,1842-10-25 09:00:00,NR,10.8,79.4,NaN,MISSING,NaN,MISSING
3,1842298N11080,1842,NI,BB,UNNAMED,1842-10-25 12:00:00,NR,10.8,78.9,NaN,MISSING,NaN,MISSING
4,1842298N11080,1842,NI,BB,UNNAMED,1842-10-25 15:00:00,NR,10.8,78.4,NaN,MISSING,NaN,MISSING
5,1842298N11080,1842,NI,AS,UNNAMED,1842-10-25 18:00:00,NR,10.8,77.9,NaN,MISSING,NaN,MISSING
6,1842298N11080,1842,NI,AS,UNNAMED,1842-10-25 21:00:00,NR,10.8,77.4,NaN,MISSING,NaN,MISSING
7,1842298N11080,1842,NI,AS,UNNAMED,1842-10-26 00:00:00,NR,10.8,76.9,NaN,MISSING,NaN,MISSING
8,1842298N11080,1842,NI,AS,UNNAMED,1842-10-26 03:00:00,NR,10.8,76.4,NaN,MISSING,NaN,MISSING
9,1842298N11080,1842,NI,AS,UNNAMED,1842-10-26 06:00:00,NR,10.8,75.8,NaN,MISSING,NaN,MISSING


In [32]:
# 28. SAVE FULL CLEANED DATASET

CLEANED_DIR.mkdir(parents=True, exist_ok=True)

full_output = CLEANED_DIR / "ibtracs_NI_cleaned.csv"

clean_tracks.to_csv(
    full_output,
    index=False
)

print("Full cleaned dataset saved to:")
print(full_output)

Full cleaned dataset saved to:
d:\SIH\tropical-cyclone-ai\data\interim\cleaned_tracks\ibtracs_NI_cleaned.csv


In [33]:
# 29. CREATE TCIR PERIOD SUBSET

tcir_tracks = clean_tracks[
    clean_tracks["SEASON"].between(2003, 2016)
].copy()

print("TCIR-period observations:", len(tcir_tracks))
print("TCIR-period cyclones:", tcir_tracks["SID"].nunique())

TCIR-period observations: 5103
TCIR-period cyclones: 132


In [34]:
# 30. SAVE TCIR PERIOD SUBSET

tcir_output = CLEANED_DIR / "ibtracs_NI_2003_2016.csv"

tcir_tracks.to_csv(
    tcir_output,
    index=False
)

print("TCIR-period dataset saved to:")
print(tcir_output)

TCIR-period dataset saved to:
d:\SIH\tropical-cyclone-ai\data\interim\cleaned_tracks\ibtracs_NI_2003_2016.csv


In [35]:
# 31. FINAL VERIFICATION

print("Full cleaned dataset:")
print("  Rows:", len(clean_tracks))
print("  Columns:", len(clean_tracks.columns))

print("\nTCIR-period dataset:")
print("  Rows:", len(tcir_tracks))
print("  Cyclones:", tcir_tracks["SID"].nunique())

print("\nFiles created:")
print(" ", full_output)
print(" ", tcir_output)

Full cleaned dataset:
  Rows: 62848
  Columns: 13

TCIR-period dataset:
  Rows: 5103
  Cyclones: 132

Files created:
  d:\SIH\tropical-cyclone-ai\data\interim\cleaned_tracks\ibtracs_NI_cleaned.csv
  d:\SIH\tropical-cyclone-ai\data\interim\cleaned_tracks\ibtracs_NI_2003_2016.csv


# 03 Data Cleaning — Summary

This notebook cleaned and standardized the IBTrACS North Indian Ocean cyclone track data.

## Cleaning performed
- Loaded the raw IBTrACS dataset without modifying the original file.
- Created a separate working copy for cleaning.
- Standardized column names and missing values.
- Converted dates, coordinates, season, wind, and pressure fields to appropriate data types.
- Cleaned important text fields.
- Removed rows missing essential track information.
- Validated latitude and longitude ranges.
- Checked and removed duplicate cyclone-time observations.
- Checked wind and pressure value ranges.
- Created canonical `WIND_KTS` and `PRESSURE_MB` fields using available agency sources.
- Recorded the source of each canonical intensity value.
- Selected the required track fields and sorted observations by cyclone and time.

## Output datasets
Two cleaned datasets were created:
- Full cleaned NI IBTrACS dataset.
- 2003–2016 subset for compatibility with the TCIR satellite dataset.

The original raw IBTrACS files remain unchanged.

## Next step
The cleaned track data will be used together with the satellite dataset during preprocessing and data fusion.